In [1]:
import numpy as np
import pandas as pd
from typing import List, Optional
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve, confusion_matrix, classification_report
)
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt

In [2]:
# df = pd.read_csv(
#     "/Users/alex._choo/Desktop/CAP5771/Loan DS/accepted_2007_to_2018Q4.csv/accepted_2007_to_2018Q4.csv",
#     low_memory=False
# )

# df.head()

columns_to_keep = [
    "id", "loan_status",
    "annual_inc", "emp_length", "home_ownership", "verification_status",
    "fico_range_low", "fico_range_high", "earliest_cr_line",
    "open_acc", "total_acc", "delinq_2yrs", "pub_rec",
    "dti", "revol_bal", "revol_util",
    "loan_amnt", "term", "int_rate", "installment",
    "purpose", "grade", "sub_grade",
    "issue_d", "addr_state"
]

valid_status = [
    "Fully Paid",
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Fully Paid",
    "Does not meet the credit policy. Status:Charged Off"
]

chunk_iter = pd.read_csv(
    "/Users/alex._choo/Desktop/CAP5771/Loan DS/accepted_2007_to_2018Q4.csv/accepted_2007_to_2018Q4.csv",
    usecols=columns_to_keep,
    chunksize=200_000,
    low_memory=False
)

clean_chunks = []

for chunk in chunk_iter:
    chunk.columns = chunk.columns.str.strip()
    
    # Rename id → loan_id
    if "id" in chunk.columns:
        chunk = chunk.rename(columns={"id": "loan_id"})

    chunk = chunk[chunk["loan_status"].isin(valid_status)]
    
    clean_chunks.append(chunk)

df_clean = pd.concat(clean_chunks, ignore_index=True)

print(df_clean.shape)

(1348099, 25)


In [3]:
display(df_clean.describe().T)

,count,mean,std,min,25%,50%,75%,max
loan_amnt,1348099.0,14408.998913,8716.137925,500.00,7975.00,12000.00,20000.00,40000.00
int_rate,1348099.0,13.241562,4.765685,5.31,9.75,12.74,15.99,30.99
installment,1348099.0,437.777843,261.497190,4.93,248.28,375.04,580.22,1719.83
annual_inc,1348095.0,76237.743295,69922.741975,0.00,45750.00,65000.00,90000.00,10999200.00
dti,1347725.0,18.274253,11.155495,-1.00,11.79,17.61,24.05,999.00
delinq_2yrs,1348070.0,0.317633,0.877744,0.00,0.00,0.00,0.00,39.00
fico_range_low,1348099.0,696.162233,31.850787,610.00,670.00,690.00,710.00,845.00
fico_range_high,1348099.0,700.162371,31.851434,614.00,674.00,694.00,714.00,850.00
open_acc,1348070.0,11.590447,5.474680,0.00,8.00,11.00,14.00,90.00
pub_rec,1348070.0,0.215050,0.601472,0.00,0.00,0.00,0.00,86.00


In [4]:
df_clean["loan_status"].value_counts()

loan_status
Fully Paid                                             1076751
Charged Off                                             268559
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

In [5]:
status_map = {
    "Fully Paid": 0,
    "Does not meet the credit policy. Status:Fully Paid": 0,
    "Charged Off": 1,
    "Does not meet the credit policy. Status:Charged Off": 1,
    "Default": 1
}

df_clean["target"] = df_clean["loan_status"].map(status_map)

# Safety check
print(df_clean["target"].value_counts())

target
0    1078739
1     269360
Name: count, dtype: int64


In [6]:
df_model_full = df_clean.copy()  # includes grade/int_rate

df_model_fundamental = df_clean.drop(
    columns=["grade", "sub_grade", "int_rate", "installment"]
)

In [7]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

# Drop Duplicates
class DropDuplicatesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, subset: Optional[List[str]] = None):
        self.subset = subset or ["loan_id"]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # only drop duplicates if all subset columns exist
        if all(col in X.columns for col in self.subset):
            return X.drop_duplicates(subset=self.subset).reset_index(drop=True)
        # If subset not present, do nothing (safe)
        return X.reset_index(drop=True)
    
# Parse Dates
class ParseDatesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, date_cols: List[str]):
        self.date_cols = date_cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.date_cols:
            if c in X.columns:
                X[c] = pd.to_datetime(X[c], errors="coerce")
        return X

# Convert Percent Strings to Float
class PercentToFloatTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cols: List[str]):
        self.cols = cols
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for c in self.cols:
            if c in X.columns:
                X[c] = X[c].astype(str).str.replace('%','',regex=False).str.replace(',', '', regex=False).str.strip()
                X[c] = pd.to_numeric(X[c], errors='coerce')
                if c == "revol_util":
                    X[c] = X[c].where(X[c] >= 0, np.nan)  # negative->NaN defensively
                    X[c] = X[c].clip(lower=0, upper=100)
        return X
    
# Extract Loan Term (36 / 60)
class TermExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, col="term"):
        self.col = col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.col in X.columns:
            X[self.col] = (
                X[self.col]
                .astype(str)
                .str.extract(r"(\d+)")[0]
            )
            X[self.col] = pd.to_numeric(X[self.col], errors='coerce')
        return X
    
# Convert Employment Length to Numeric
class EmpLengthTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, col="emp_length"):
        self.col = col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.col in X.columns:
            s = X[self.col].astype(str).str.lower().str.strip()
            # normalize common tokens to NaN first
            s = s.replace({"n/a": np.nan, "na": np.nan, "none": np.nan, "unknown": np.nan, "nan": np.nan})
            # '10+' -> 10, '< 1' -> 0
            s = s.str.replace(r"10\+", "10", regex=True)
            s = s.str.replace(r"<\s*1", "0", regex=True)
            # extract first number
            nums = s.str.extract(r"(\d+)", expand=False)
            X[self.col + "_years"] = pd.to_numeric(nums, errors="coerce")
        return X
    
# Engineer FICO + Credit Age
class FicoAndCreditAgeTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if "fico_range_low" in X.columns and "fico_range_high" in X.columns:
            X["fico_range_low"] = pd.to_numeric(X["fico_range_low"], errors="coerce")
            X["fico_range_high"] = pd.to_numeric(X["fico_range_high"], errors="coerce")
            X["fico"] = (X["fico_range_low"] + X["fico_range_high"]) / 2

        if "issue_d" in X.columns and "earliest_cr_line" in X.columns:
            X["issue_d"] = pd.to_datetime(X["issue_d"], errors="coerce")
            X["earliest_cr_line"] = pd.to_datetime(X["earliest_cr_line"], errors="coerce")
            X["credit_age_years"] = (
                (X["issue_d"] - X["earliest_cr_line"]).dt.days / 365
            )

        return X

# Drop Unneeded Columns
class DropColumnsTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cols: List[str]):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        existing = [c for c in self.cols if c in X.columns]
        return X.drop(columns=existing)

# Clean Categorical Variables
class CategoricalCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, cols: List[str], rare_thresh=0.01):
        self.cols = cols
        self.rare_thresh = rare_thresh
        self.keep_categories_ = {}

    def fit(self, X, y=None):
        for col in self.cols:
            if col in X.columns:
                s = X[col].copy()
                # normalize strings while preserving NaN
                s = s.where(pd.notna(s), other=np.nan)
                s = s.astype(str).str.strip().str.lower()
                # convert 'nan' strings back to np.nan (if any)
                s = s.replace({'nan': np.nan})
                vc = s.value_counts(normalize=True, dropna=True)
                top = vc[vc >= self.rare_thresh].index.tolist()
                self.keep_categories_[col] = set(top)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            if col in X.columns:
                s = X[col].copy()
                s = s.where(pd.notna(s), other=np.nan)
                s = s.astype(str).str.strip().str.lower()
                s = s.replace({'nan': np.nan})
                allowed = self.keep_categories_.get(col, None)
                if allowed is not None:
                    # keep NaN as NaN
                    s = s.where(s.isin(allowed), other="__other__")
                X[col] = s
        return X
    
# Winsorize Numeric Outliers
class NumericWinsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, cols: List[str], lower_q: float = 0.01, upper_q: float = 0.99):
        self.cols = cols
        self.lower_q = lower_q
        self.upper_q = upper_q
        self.bounds_ = {}

    def fit(self, X, y=None):
        X = X.copy()
        for col in self.cols:
            if col in X.columns:
                s = pd.to_numeric(X[col], errors="coerce")
                lo = s.quantile(self.lower_q)
                hi = s.quantile(self.upper_q)
                # if lo or hi is NaN (all missing), skip
                if pd.isna(lo) or pd.isna(hi):
                    continue
                self.bounds_[col] = (lo, hi)
        return self

    def transform(self, X):
        X = X.copy()
        for col, (lo, hi) in self.bounds_.items():
            if col in X.columns:
                # coerce to numeric then clip
                X[col] = pd.to_numeric(X[col], errors="coerce")
                X[col] = X[col].clip(lower=lo, upper=hi)
                X[col] = X[col].where(X[col] >= 0, np.nan)  # negative->NaN defensively
        return X
    
# Missing Indicators + Median Imputation
class MissingIndicatorAndMedianImputer(BaseEstimator, TransformerMixin):
    def __init__(self, numeric_cols: List[str]):
        self.numeric_cols = numeric_cols
        self.medians_ = {}

    def fit(self, X, y=None):
        for col in self.numeric_cols:
            if col in X.columns:
                self.medians_[col] = pd.to_numeric(
                    X[col], errors="coerce"
                ).median()
        return self

    def transform(self, X):
        X = X.copy()
        for col, median in self.medians_.items():
            if col in X.columns:
                X[col + "_missing"] = pd.to_numeric(
                    X[col], errors="coerce"
                ).isna().astype(int)
                X[col] = pd.to_numeric(
                    X[col], errors="coerce"
                ).fillna(median)
        return X
    
# Log Transform Skewed Numeric Features
class LogTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p to specified numeric columns (coerce non-numeric -> NaN)."""
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cols:
            if c in X.columns:
                X[c] = np.log1p(pd.to_numeric(X[c], errors="coerce"))
        return X


In [8]:
def build_cleaning_pipeline(numeric_cols: List[str], categorical_cols: List[str],
                            log_cols: Optional[List[str]] = None):
    steps = []
    steps.append(("drop_duplicates", DropDuplicatesTransformer(subset=["loan_id"])))
    steps.append(("parse_dates", ParseDatesTransformer(date_cols=["issue_d", "earliest_cr_line"])))
    steps.append(("percent_to_float", PercentToFloatTransformer(cols=["int_rate", "revol_util"])))
    steps.append(("term_extract", TermExtractor(col="term")))
    steps.append(("emp_len", EmpLengthTransformer(col="emp_length")))
    steps.append(("fico_credit", FicoAndCreditAgeTransformer()))

    steps.append(("drop_raw_fico", DropColumnsTransformer(cols=["fico_range_low", "fico_range_high"])))

    steps.append(("cat_clean", CategoricalCleaner(cols=categorical_cols, rare_thresh=0.01)))

    # Insert log transform here (before winsorizing)
    if log_cols:
        steps.append(("log_transform", LogTransformer(cols=log_cols)))

    steps.append(("winsorize", NumericWinsorizer(cols=numeric_cols, lower_q=0.01, upper_q=0.99)))
    steps.append(("impute", MissingIndicatorAndMedianImputer(numeric_cols=numeric_cols)))
    
    return Pipeline(steps)

In [9]:
numeric_cols = [
    "annual_inc","open_acc","total_acc","delinq_2yrs","pub_rec","dti",
    "revol_bal","revol_util","loan_amnt","int_rate","installment",
    "fico", "credit_age_years", "emp_length_years"
]
categorical_cols = ["home_ownership","verification_status","purpose","grade","sub_grade","addr_state"]

# Log-transform these heavy-tailed numeric columns BEFORE winsorize:
log_cols = ["annual_inc", "revol_bal", "loan_amnt"]

In [10]:
df_clean["issue_d"] = pd.to_datetime(df_clean["issue_d"], errors="coerce")
df_clean = df_clean.sort_values("issue_d").reset_index(drop=True)

# example split years (adjust to your preference)
train_df = df_clean[df_clean["issue_d"].dt.year <= 2015].copy()
val_df   = df_clean[df_clean["issue_d"].dt.year == 2016].copy()
test_df  = df_clean[df_clean["issue_d"].dt.year >= 2017].copy()

print("rows total / train / val / test:", len(df_clean), len(train_df), len(val_df), len(test_df))

/var/folders/3f/ch7gfc9n2k753zy2x2ds4pgm0000gn/T/ipykernel_19107/3095069245.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["issue_d"] = pd.to_datetime(df_clean["issue_d"], errors="coerce")


rows total / train / val / test: 1348099 829355 293105 225639


In [11]:
# Log-transform version for logistic regression
log_cols = ["annual_inc", "revol_bal", "loan_amnt"]
pipeline_logistic = build_cleaning_pipeline(
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    log_cols=log_cols
)

# No log-transform version for XGBoost 
pipeline_xgb = build_cleaning_pipeline(
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    log_cols=None  # no log needed
)

In [12]:
train_fund = train_df.drop(columns=["grade","sub_grade","int_rate","installment"], errors='ignore').copy()
val_fund   = val_df.drop(columns=["grade","sub_grade","int_rate","installment"], errors='ignore').copy()
test_fund  = test_df.drop(columns=["grade","sub_grade","int_rate","installment"], errors='ignore').copy()

# full (keep grade/int_rate)
train_full = train_df.copy()
val_full   = val_df.copy()
test_full  = test_df.copy()

# target vectors
y_train_f = train_fund["target"].astype(int).copy()
y_val_f   = val_fund["target"].astype(int).copy()
y_test_f  = test_fund["target"].astype(int).copy()

y_train_full = train_full["target"].astype(int).copy()
y_val_full   = val_full["target"].astype(int).copy()
y_test_full  = test_full["target"].astype(int).copy()

# drop target column from feature sets
for df in (train_fund, val_fund, test_fund, train_full, val_full, test_full):
    if "target" in df.columns:
        df.drop(columns=["target"], inplace=True)
    if "loan_id" in df.columns:
        df.drop(columns=["loan_id"], inplace=True)   # prevent accidental leakage

In [13]:
# logistic cleaning pipeline (with log transforms)
pipeline_logistic.fit(train_fund)

# xgb cleaning pipeline (no log transforms)
pipeline_xgb.fit(train_fund)  # fit on same raw features; we'll use pipeline_xgb later for both fundamental/full (it will drop/keep columns as present)

# transform all splits
X_train_fund_clean = pipeline_logistic.transform(train_fund)
X_val_fund_clean   = pipeline_logistic.transform(val_fund)
X_test_fund_clean  = pipeline_logistic.transform(test_fund)

# For XGBoost we want numeric versions too. Use pipeline_xgb (no log) for xgb-prepped features
X_train_fund_xgb = pipeline_xgb.transform(train_fund)
X_val_fund_xgb   = pipeline_xgb.transform(val_fund)
X_test_fund_xgb  = pipeline_xgb.transform(test_fund)

# For full-feature XGB (grade + int_rate present), transform using pipeline_xgb fitted to FUND training.
# Important: pipeline_xgb learned category tops / medians on train_fund. If you want pipeline_xgb to learn top categories for grade/int_rate,
# you could also refit pipeline_xgb on train_full. That's reasonable for the FULL model:
pipeline_xgb_full = build_cleaning_pipeline(numeric_cols=numeric_cols, categorical_cols=categorical_cols, log_cols=None)
pipeline_xgb_full.fit(train_full)
X_train_full_xgb = pipeline_xgb_full.transform(train_full)
X_val_full_xgb   = pipeline_xgb_full.transform(val_full)
X_test_full_xgb  = pipeline_xgb_full.transform(test_full)

/var/folders/3f/ch7gfc9n2k753zy2x2ds4pgm0000gn/T/ipykernel_19107/3817507865.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X[c] = pd.to_datetime(X[c], errors="coerce")
/var/folders/3f/ch7gfc9n2k753zy2x2ds4pgm0000gn/T/ipykernel_19107/3817507865.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X[c] = pd.to_datetime(X[c], errors="coerce")
/var/folders/3f/ch7gfc9n2k753zy2x2ds4pgm0000gn/T/ipykernel_19107/3817507865.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X[c] = pd.to_datetime(X[c], errors="coerce")
/var/folders/3f/ch7gfc9n2k753zy2x2ds4pgm0000gn/T/ipykernel

In [14]:
# After fitting pipeline_logistic on train_fund
cleaned_sample = pipeline_logistic.transform(train_fund.head(5))
print(cleaned_sample.dtypes)
print(cleaned_sample.head())

# Check medians / bounds were learned
print("Winsor bounds:", pipeline_logistic.named_steps['winsorize'].bounds_)
print("Medians:", pipeline_logistic.named_steps['impute'].medians_)

# Ensure no 'loan_id' related error
_ = pipeline_logistic.transform(val_fund.head(1))  # should not raise KeyError

loan_amnt                          float64
term                                 int64
emp_length                          object
home_ownership                      object
annual_inc                         float64
verification_status                 object
issue_d                     datetime64[ns]
loan_status                         object
purpose                             object
addr_state                          object
dti                                float64
delinq_2yrs                        float64
earliest_cr_line            datetime64[ns]
open_acc                           float64
pub_rec                            float64
revol_bal                          float64
revol_util                         float64
total_acc                          float64
emp_length_years                     int64
fico                               float64
credit_age_years                   float64
annual_inc_missing                   int64
open_acc_missing                     int64
total_acc_m

/var/folders/3f/ch7gfc9n2k753zy2x2ds4pgm0000gn/T/ipykernel_19107/3817507865.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X[c] = pd.to_datetime(X[c], errors="coerce")


In [15]:
miss_pct = df_clean.isna().mean().sort_values(ascending=False)
print(miss_pct.head(30))   # quantify

emp_length             0.058267
revol_util             0.000665
dti                    0.000277
total_acc              0.000022
pub_rec                0.000022
open_acc               0.000022
earliest_cr_line       0.000022
delinq_2yrs            0.000022
annual_inc             0.000003
loan_id                0.000000
addr_state             0.000000
revol_bal              0.000000
fico_range_high        0.000000
fico_range_low         0.000000
purpose                0.000000
loan_amnt              0.000000
loan_status            0.000000
issue_d                0.000000
verification_status    0.000000
home_ownership         0.000000
sub_grade              0.000000
grade                  0.000000
installment            0.000000
int_rate               0.000000
term                   0.000000
target                 0.000000
dtype: float64


In [16]:
print("Shape:", df_clean.shape)
print("\nColumns:\n", df_clean.columns.tolist())
print("\nData types:\n", df_clean.dtypes)

Shape: (1348099, 26)

Columns:
 ['loan_id', 'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'target']

Data types:
 loan_id                        object
loan_amnt                     float64
term                           object
int_rate                      float64
installment                   float64
grade                          object
sub_grade                      object
emp_length                     object
home_ownership                 object
annual_inc                    float64
verification_status            object
issue_d                datetime64[ns]
loan_status                    object
purpose                        object
addr_state                     object
dti                           

In [17]:
print(df_clean["target"].value_counts(normalize=True))

target
0    0.800193
1    0.199807
Name: proportion, dtype: float64


In [18]:
df_clean.describe(percentiles=[0.01,0.5,0.99]).T

,count,mean,min,1%,50%,99%,max,std
loan_amnt,1348099.0,14408.998913,500.0,1500.0,12000.0,35000.0,40000.0,8716.137925
int_rate,1348099.0,13.241562,5.31,5.32,12.74,26.3,30.99,4.765685
installment,1348099.0,437.777843,4.93,52.75,375.04,1220.33,1719.83,261.49719
annual_inc,1348095.0,76237.743295,0.0,18000.0,65000.0,251000.0,10999200.0,69922.741975
issue_d,1348099,2015-06-02 04:04:23.848871168,2007-06-01 00:00:00,2010-07-01 00:00:00,2015-08-01 00:00:00,2018-07-01 00:00:00,2018-12-01 00:00:00,NaN
dti,1347725.0,18.274253,-1.0,1.77,17.61,38.47,999.0,11.155495
delinq_2yrs,1348070.0,0.317633,0.0,0.0,0.0,4.0,39.0,0.877744
fico_range_low,1348099.0,696.162233,610.0,660.0,690.0,800.0,845.0,31.850787
fico_range_high,1348099.0,700.162371,614.0,664.0,694.0,804.0,850.0,31.851434
open_acc,1348070.0,11.590447,0.0,3.0,11.0,29.0,90.0,5.47468


In [19]:
print("Negative income:", (df_clean["annual_inc"] < 0).sum())
print("DTI < 0:", (df_clean["dti"] < 0).sum())
print("revol_util > 100:", (df_clean["revol_util"] > 100).sum())
print("FICO < 300:", (df_clean["fico_range_low"] < 300).sum())

Negative income: 0
DTI < 0: 2
revol_util > 100: 4714
FICO < 300: 0


In [20]:
print("Missing issue_d:", df_clean["issue_d"].isna().sum())
print("Missing earliest_cr_line:", df_clean["earliest_cr_line"].isna().sum())

Missing issue_d: 0
Missing earliest_cr_line: 29


In [21]:
df_test_cleaned = pipeline_logistic.fit_transform(train_fund.head(1000))

/var/folders/3f/ch7gfc9n2k753zy2x2ds4pgm0000gn/T/ipykernel_19107/3817507865.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X[c] = pd.to_datetime(X[c], errors="coerce")


In [22]:
print(df_test_cleaned.isna().sum().sum())
print(df_test_cleaned.describe().T.head())

29
             count                        mean                  min  \
loan_amnt   1000.0                    8.791302             6.398595   
term        1000.0                        36.0                 36.0   
annual_inc  1000.0                   10.783595             8.699681   
issue_d       1000  2007-11-08 12:17:16.800000  2007-06-01 00:00:00   
dti         1000.0                    11.40163                  0.0   

                            25%                  50%                  75%  \
loan_amnt                8.2943             8.853808             9.392745   
term                       36.0                 36.0                 36.0   
annual_inc             10.35698            10.819798            11.225257   
issue_d     2007-10-01 00:00:00  2007-12-01 00:00:00  2008-01-01 00:00:00   
dti                      5.1125               10.825              17.0325   

                            max       std  
loan_amnt             10.126671  0.812888  
term               

In [23]:
def quick_health(df):
    print("Rows:", len(df))
    print("Duplicates:", df.duplicated().sum())
    print("Total Missing %:", round(df.isna().mean().mean()*100,2), "%")
    print("Numeric columns:", len(df.select_dtypes(include='number').columns))
    print("Object columns:", len(df.select_dtypes(include='object').columns))

quick_health(df_clean)

Rows: 1348099
Duplicates: 0
Total Missing %: 0.23 %
Numeric columns: 14
Object columns: 11


In [24]:
df_clean.to_csv("analysis_dataset.csv", index=False)